# Pha R (mo rong) — Buoc 1: Sinh corpus tu du lieu PHI TUYEN (periodic coupling)

**Cau hoi dat ra:** Pha R goc (VAR tuyen tinh Gaussian) cho thay KSG thang ro o N=10,20. Nhung KSG co the chi manh tren dung 'san nha' cua no (phan phoi Gaussian tron, low-dimensional). Vong nay lap lai TOAN BO quy trinh danh gia tren du lieu **ghep noi tuan hoan** (`periodic_coupling.py`, mo phong gan hon voi tim-nao thuc te so voi VAR) de xem ket luan co doi khong.

**Luoi nho hon** ban goc (4 coupling x 2 noise x 6 N x 200 = 9.600 cua so train+val) de kiem soat thoi gian cho vong thu nghiem dau tien.

**Output:** `data/interim/synthetic_corpus_periodic/`.

Xem `docs/PHASE_R_REPORT.md` muc 9 (phuong an b) va `configs/synthetic/corpus_periodic.yaml`.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import yaml, numpy as np
from pathlib import Path

from pqrst.data.synthetic.corpus import (generate_corpus_periodic, split_corpus_by_seed,
                                         save_corpus, load_corpus)

BASE = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
cfg = yaml.safe_load(open(BASE/'configs'/'synthetic'/'corpus_periodic.yaml', encoding='utf-8'))
pgt = {(d['c'], d['noise_std']): d['te'] for d in cfg['pseudo_ground_truths']}
cfg

{'omega_x': 0.3,
 'omega_y': 0.31,
 'coupling_values': [0.0, 0.3, 0.6, 0.9],
 'noise_values': [0.1, 0.3],
 'n_values': [10, 20, 30, 50, 100, 200],
 'n_windows_per_cell': 200,
 'pseudo_ground_truths': [{'c': 0.0, 'noise_std': 0.1, 'te': 0.0},
  {'c': 0.0, 'noise_std': 0.3, 'te': 0.0},
  {'c': 0.3, 'noise_std': 0.1, 'te': 0.13598},
  {'c': 0.3, 'noise_std': 0.3, 'te': 0.035753},
  {'c': 0.6, 'noise_std': 0.1, 'te': 0.249306},
  {'c': 0.6, 'noise_std': 0.3, 'te': 0.108777},
  {'c': 0.9, 'noise_std': 0.1, 'te': 0.36413},
  {'c': 0.9, 'noise_std': 0.3, 'te': 0.197035}],
 'base_seed': 2000000,
 'val_fraction': 0.15,
 'split_seed': 11,
 'test_base_seed': 2900000,
 'test_n_windows_per_cell': 150}

## 1. Sinh corpus train/validation

In [2]:
import time
t0 = time.time()
corpus = generate_corpus_periodic(
    coupling_values=cfg['coupling_values'], noise_values=cfg['noise_values'],
    n_values=cfg['n_values'], n_windows_per_cell=cfg['n_windows_per_cell'],
    omega_x=cfg['omega_x'], omega_y=cfg['omega_y'],
    pseudo_ground_truths=pgt, base_seed=cfg['base_seed'])
print(f'Time: {time.time()-t0:.2f}s')
print(f'Tong so cua so: {len(corpus)}')
from collections import Counter
print('Phan bo N:', Counter([w.n_samples for w in corpus]))

Time: 3.42s
Tong so cua so: 9600
Phan bo N: Counter({10: 1600, 20: 1600, 30: 1600, 50: 1600, 100: 1600, 200: 1600})


## 2. Chia train/validation (theo cua so)

In [3]:
train_windows, val_windows = split_corpus_by_seed(
    corpus, val_fraction=cfg['val_fraction'], split_seed=cfg['split_seed'])
print(f'Train: {len(train_windows)}, Val: {len(val_windows)}')

Train: 8160, Val: 1440


## 3. Sinh tap TEST rieng biet (seed khac hoan toan)

In [4]:
test_corpus = generate_corpus_periodic(
    coupling_values=cfg['coupling_values'], noise_values=cfg['noise_values'],
    n_values=cfg['n_values'], n_windows_per_cell=cfg['test_n_windows_per_cell'],
    omega_x=cfg['omega_x'], omega_y=cfg['omega_y'],
    pseudo_ground_truths=pgt, base_seed=cfg['test_base_seed'])
print(f'Test size: {len(test_corpus)}')

Test size: 7200


## 4. Luu ra dia + kiem tra round-trip

In [5]:
out_dir = BASE / 'data' / 'interim' / 'synthetic_corpus_periodic'
out_dir.mkdir(parents=True, exist_ok=True)
save_corpus(train_windows, str(out_dir / 'train.npz'))
save_corpus(val_windows, str(out_dir / 'val.npz'))
save_corpus(test_corpus, str(out_dir / 'test.npz'))

loaded = load_corpus(str(out_dir / 'train.npz'))
assert len(loaded) == len(train_windows)
assert np.array_equal(loaded[0].y_t, train_windows[0].y_t)
print('Round-trip OK.')

Round-trip OK.


## 5. Sanity check — TE tang theo coupling

In [6]:
import pandas as pd
rows = [{'c': w.params['c'], 'noise': w.params['noise_std'], 'te_gt': w.te_ground_truth}
        for w in test_corpus]
df = pd.DataFrame(rows).drop_duplicates().sort_values(['noise', 'c'])
display(df)
assert (df.sort_values('c').groupby('noise')['te_gt'].apply(lambda s: s.is_monotonic_increasing)).all()
print('TE tang don dieu theo coupling: OK')

,c,noise,te_gt
0,0.0,0.1,0.000000
1800,0.3,0.1,0.135980
3600,0.6,0.1,0.249306
5400,0.9,0.1,0.364130
900,0.0,0.3,0.000000
2700,0.3,0.3,0.035753
4500,0.6,0.3,0.108777
6300,0.9,0.3,0.197035


TE tang don dieu theo coupling: OK
